# Batch product scraping

Input CSV → one artifact per row → output mapping CSV. Uses Crawl4AI multi-profile capture, strict capture decisions, and same-run domain profile learning.


In [ ]:
from pathlib import Path
import sys, json, csv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import sys
for name in list(sys.modules):
    if name.startswith("product_scraping_agent"):
        del sys.modules[name]
print("module cache cleared")

In [ ]:
from product_scraping_agent.batch import BatchOptions, run_batch
from product_scraping_agent import ScrapeRequest
print("Batch imports OK")

## Prepare paths

In [ ]:
input_csv = PROJECT_ROOT / "data" / "samples" / "batch_input_sample.csv"
output_csv = PROJECT_ROOT / "data" / "batch_scrape_output.csv"
summary_json = PROJECT_ROOT / "data" / "batch_scrape_summary.json"
output_root = PROJECT_ROOT / "data" / "scraped"

print("input_csv:", input_csv)
print("output_csv:", output_csv)
print("summary_json:", summary_json)
list(csv.DictReader(input_csv.open(encoding="utf-8-sig")))[:5]

## Run batch

In [ ]:
summary = await run_batch(
    input_csv=input_csv,
    output_csv=output_csv,
    summary_json=summary_json,
    options=BatchOptions(
        output_root=output_root,
        max_concurrency=2,
        max_images=30,
        vision_max=12,
        max_agent_iterations=2,
        resume=False,
        domain_profile_learning=True,
    ),
)
summary.as_dict()

## Domain profile learning

When `domain_profile_learning=True`, the runner remembers the best successful Crawl4AI profile per domain during this batch. For example, if `shadow_iframe` succeeds for Amazon, later Amazon rows try `shadow_iframe` first. No search is performed and the URL is never changed.


## Inspect mapping CSV

In [ ]:
rows = list(csv.DictReader(output_csv.open(encoding="utf-8-sig")))
print("rows:", len(rows))
display_cols = [
    "input_id", "success", "artifact_quality", "requires_manual_review",
    "access_status", "browser_visible", "capture_profile_used",
    "capture_score", "capture_grade", "capture_decision", "real_scrape_evidence",
    "is_weak_capture", "is_block_or_challenge", "capture_decision_bucket",
    "image_candidate_count", "final_image_count", "artifact_dir", "error"
]
[{c: row.get(c, "") for c in display_cols} for row in rows[:20]]

## Quick anomaly view

In [ ]:
def truthy(v):
    return str(v).strip().lower() in {"true", "1", "yes"}
weak = [r for r in rows if (not truthy(r.get("real_scrape_evidence"))) or (not truthy(r.get("browser_visible"))) or truthy(r.get("requires_manual_review"))]
print("Rows needing attention:", len(weak))
[{c: row.get(c, "") for c in display_cols} for row in weak[:50]]

## Open a selected artifact report

In [ ]:
row_idx = 0
row = rows[row_idx]
for col in ["quality_report_path", "source_alignment_report_path", "metadata_json_path"]:
    path = Path(row[col])
    print("\n" + col + ":", path)
    if path.exists():
        payload = json.loads(path.read_text(encoding="utf-8"))
        print(json.dumps(payload, ensure_ascii=False, indent=2)[:4000])
    else:
        print("missing")